# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/PNRS2006/flyrank-internship-pnrs/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This notebook frames my **Urban Heat Risk Intelligence System** as a concrete ML problem before model development begins.

The project aims to identify spatial areas that are more exposed to urban heat by combining meteorological, satellite-derived, urban-form, geographical, and population-related information.

The goal of this notebook is to define the ML task, target/proxy, success metric, unit of analysis, decision supported by the output, and why a learned approach may be useful compared with a simple fixed rule.

## 1. My lane as an ML task (type)

**Task type: Classification**

My project is the **Urban Heat Risk Intelligence System**. I frame the core ML task as a classification problem because the system will assign each spatial grid cell to a heat-risk category.

The planned categories are **Very Low, Low, Moderate, High, and Very High**.

The model will use environmental and spatial signals such as temperature, humidity, land-surface temperature, vegetation indicators, built-up characteristics, road density, population exposure, and geographical variables.

The output is intended to support spatial heat-risk screening and prioritization rather than replace an official heat-health warning system. A classification for each spatial cell can later be visualized as a heat-risk map so that areas with elevated risk can be investigated first.

The initial spatial unit is planned as a **500 m × 500 m grid cell**.

In [ ]:
# Define the ML task framing

task_type = "classification"

risk_categories = [
    "Very Low",
    "Low",
    "Moderate",
    "High",
    "Very High"
]

print("Task type:", task_type)
print("Risk categories:", risk_categories)


## 2. Target or proxy

**Target: Heat-risk category for a spatial grid cell during a defined observation period.**

The target will represent the heat-risk condition of each spatial cell using the five planned categories: **Very Low, Low, Moderate, High, and Very High**.

The labels should not be assigned arbitrarily. The eventual target should be derived from observed heat intensity and exposure using a documented and reproducible method. A continuous heat-risk score can be constructed first and then converted into categories using justified thresholds.

The target is therefore a **defined proxy for spatial heat risk**, rather than a direct claim that a location is unsafe.

For a future predictive version, only information available at prediction time should be used as input. Future temperature, land-surface temperature, or other future observations must not be used as features when predicting future heat risk because that would introduce target leakage.

In [ ]:
# Define the target representation

target_name = "heat_risk_category"
target_type = "categorical"
target_source = "defined proxy based on observed heat intensity and exposure"

target_definition = {
    "name": target_name,
    "type": target_type,
    "classes": risk_categories,
    "source": target_source
}

for key, value in target_definition.items():
    print(f"{key}: {value}")


## 3. Success metric

The primary success metric will be **Recall for the High and Very High risk classes**.

This metric is important because the main purpose of the system is to screen for areas with elevated heat risk. Missing an actually high-risk area is potentially more consequential than incorrectly flagging a lower-risk area for further investigation.

I will also report **macro F1-score, precision, recall, and a confusion matrix** so that performance across all risk categories can be examined.

Accuracy alone will not be treated as sufficient because the risk classes may not be equally represented and different types of classification errors can have different practical consequences.

For a future forecasting version, evaluation should use a temporal train/validation/test split so that later observations are evaluated as genuinely unseen data.

In [ ]:
# Define the evaluation plan

primary_metric = "Recall for High and Very High risk classes"

additional_metrics = [
    "Macro F1-score",
    "Precision",
    "Recall",
    "Confusion matrix"
]

print("Primary metric:", primary_metric)
print("Additional metrics:", additional_metrics)


## 4. The unit of analysis, as a real dataframe

**One row = one spatial grid cell for one defined observation period.**

The planned spatial representation is a regular **500 m × 500 m grid** over the study area.

Each row will represent the environmental and exposure conditions associated with one grid cell at a particular observation date or period.

The intended dataset will eventually combine variables such as Grid ID, date, latitude, longitude, temperature, humidity, land-surface temperature (LST), vegetation indicators such as NDVI, built-up fraction, road density, population density, elevation, and the heat-risk target.

The dataframe below is an **illustrative schema example** used only to demonstrate the intended unit of analysis. The numerical values are not claimed to be measurements from IMD, Landsat, Sentinel-2, Census, or another external source.

In [ ]:
import pandas as pd

# Illustrative schema only.
# These values demonstrate the intended dataframe structure.
# They are NOT real environmental measurements.

unit_df = pd.DataFrame({
    "grid_id": ["G001", "G002", "G003", "G004"],
    "date": [
        "2026-05-01",
        "2026-05-01",
        "2026-05-01",
        "2026-05-01"
    ],
    "latitude": [16.50, 16.51, 16.52, 16.53],
    "longitude": [80.60, 80.61, 80.62, 80.63],
    "temperature": [38.5, 39.2, 40.1, 37.8],
    "humidity": [55, 48, 42, 61],
    "lst": [42.3, 46.1, 49.2, 40.5],
    "ndvi": [0.62, 0.31, 0.15, 0.71],
    "built_up_fraction": [0.30, 0.68, 0.86, 0.22],
    "road_density": [2.1, 4.8, 6.2, 1.7],
    "population_density": [3200, 6800, 9100, 2700],
    "heat_risk_category": [
        "Low",
        "High",
        "Very High",
        "Very Low"
    ]
})

print("Shape:", unit_df.shape)
unit_df


In [ ]:
# Verify the unit of analysis

print("One row represents: one spatial grid cell for one observation date.")
print("Number of rows:", len(unit_df))
print("Unique grid cells:", unit_df["grid_id"].nunique())
print("Unique dates:", unit_df["date"].nunique())

assert len(unit_df) == unit_df[["grid_id", "date"]].drop_duplicates().shape[0]

print("\nUnit-of-analysis check passed.")


### Why this unit makes sense

A grid-cell-based unit makes the model output directly usable as a spatial heat-risk map. If neighboring cells receive High or Very High classifications, they can be visualized as potential heat-risk hotspots.

Using spatial cells also allows the system to represent differences within the same city instead of assigning a single heat-risk value to the entire city.

The same unit can later be connected to GIS layers and dashboard components for spatial interpretation and decision support.

## 5. Why ML beats a fixed rule here

A simple fixed rule could classify a location using one threshold, such as: **if temperature is above a selected value, classify the location as high risk**.

This is easy to understand but does not fully represent urban heat risk because different locations can have similar air temperatures while having different land-surface temperatures, vegetation cover, built-up density, road density, population exposure, and geographical conditions.

A machine-learning model can combine multiple environmental, spatial, and exposure variables and learn nonlinear relationships between them. For example, high temperature combined with high LST, low vegetation, high built-up fraction, and high population density may indicate a different risk context from the same temperature in an area with greater vegetation and lower exposure.

However, ML should not automatically be assumed to be better than a rule. I would first establish a transparent rule-based baseline and compare the ML model against it using the chosen evaluation metrics.

The final model output is intended to support human decision-making by identifying spatial areas that deserve further heat-risk investigation or possible cooling and adaptation interventions.

In [ ]:
# Illustrative fixed-rule baseline.
# This is only used to demonstrate the type of simple rule
# that the eventual ML approach should be compared against.

temperature_threshold = 40.0

unit_df["rule_based_high_risk_flag"] = (
    unit_df["temperature"] >= temperature_threshold
)

unit_df[[
    "grid_id",
    "temperature",
    "heat_risk_category",
    "rule_based_high_risk_flag"
]]


## Self-check

- [x] Every section above is filled — markdown thinking and the code that backs it.
- [x] The ML task is explicitly defined as classification.
- [x] The target/proxy is explicitly defined as a heat-risk category.
- [x] The target is described as a defined proxy based on observed heat intensity and exposure rather than an arbitrary label.
- [x] The primary success metric is recall for High and Very High risk classes.
- [x] Additional evaluation metrics are identified.
- [x] The unit of analysis is explicit: one row = one spatial grid cell for one observation period.
- [x] A dataframe demonstrates the intended unit of analysis.
- [x] The dataframe is clearly labelled as illustrative and does not pretend to contain externally sourced measurements.
- [x] The output supports a real decision: spatial heat-risk screening and intervention prioritization.
- [x] A simple fixed-rule baseline is included for comparison.
- [x] Future observations are identified as a potential source of target leakage.
- [x] The system is described as decision-support/risk-screening rather than an official heat-health warning system.
- [x] No client names, private queries, or confidential information are included.
- [x] The notebook is intended to be committed under `work/notebooks/w02_ml_task_framing.ipynb`.